# Visualize buggy attention with Treescope

[Treescope](https://github.com/google-deepmind/treescope) pretty-prints arrays and other large Python objects in a notebook, which can help with inspecting intermediate values or model weights.

The following example is based on real bugs ([PR #4016](https://github.com/AI-Hypercomputer/maxtext/pull/4016), [issue #4476](https://github.com/AI-Hypercomputer/maxtext/issues/4476)) where Treescope could have helped visualize the problem.

> Context: To avoid wasting compute on padding, training packs several short examples into one row, and each token must only attend to its own example. MaxText enforces this with segment ids. If they are lost (e.g., if you call the model without them, or the attention kernel ignores them), attention silently leaks across examples.

Such a bug can be hard to diagnose just by looking at logits or the loss curve, since nothing crashes and the loss still goes down. But if you inspect the attention mask using Treescope, it is immediately obvious.

## Setup

Get a notebook running with MaxText by following [Run MaxText Python Notebooks on TPUs](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html).

In [ ]:
import os

import jax
import numpy as np
import transformers
import treescope
from flax import nnx

from maxtext.configs import pyconfig
from maxtext.utils import model_creation_utils
from maxtext.utils.globals import MAXTEXT_PKG_DIR

treescope.basic_interactive_setup(autovisualize_arrays=True)

## Pack three examples into one row

Each example gets its own segment id, and its positions start again from 0.

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
examples = ["The cat sat on the mat.", "Paris is the capital of France.", "Two plus two is four."]
pieces = [tokenizer(text)["input_ids"] for text in examples]

copies = (jax.device_count(), 1)  # MaxText runs one copy of the row on each device
tokens = np.tile(np.concatenate(pieces), copies)
segment_ids = np.tile(np.concatenate([np.full(len(p), i + 1) for i, p in enumerate(pieces)]), copies)
positions = np.tile(np.concatenate([np.arange(len(p)) for p in pieces]), copies)
print("segment ids:", segment_ids[0])
print("positions:  ", positions[0])

## Run the model and record intermediates

Create a small Qwen3 model (2 layers, random weights). We'll use Flax's [`nnx.capture`](https://flax.readthedocs.io/en/latest/api_reference/flax.nnx/module.html) to record the intermediate results of a forward pass, and run the model once with segment ids and once without.

In [ ]:
config = pyconfig.initialize(
    [None, os.path.join(MAXTEXT_PKG_DIR, "configs", "base.yml")],
    model_name="qwen3-0.6b",
    override_model_config=True,
    base_num_decoder_layers=2,
    per_device_batch_size=1,
    max_target_length=tokens.shape[1],
    attention="dot_product",  # the plain attention implementation, which builds the mask as an array
    skip_jax_distributed_system=True,
)
model, _ = model_creation_utils.from_pretrained(config)

capture = nnx.capture(model, nnx.Intermediate, method_outputs=nnx.Intermediate)
_, with_ids = capture(decoder_input_tokens=tokens, decoder_positions=positions, decoder_segment_ids=segment_ids)
_, without_ids = capture(decoder_input_tokens=tokens, decoder_positions=positions)

## Look at the recorded data with Treescope

Treescope automatically renders notebook outputs as a tree. Click ▶ to open parts of it. The attention mask is under `decoder > layers > self_attention > attention_op > generate_attention_mask`.

In [ ]:
with_ids

Compare that mask with the one from the run without segment ids, for the first layer and first row. Row *i* shows which tokens token *i* may attend to: light cells are allowed and dark cells are blocked.

In [ ]:
attention = lambda recorded: recorded["decoder"]["layers"]["self_attention"]["attention_op"]
{
    "with segment ids": attention(with_ids)["generate_attention_mask"].get_value()[0][0, 0, 0, 0],
    "without segment ids": attention(without_ids)["generate_attention_mask"].get_value()[0][0, 0, 0, 0],
}

With segment ids, the mask is three separate triangles, one per example. Without them, it is one big triangle: every token can also attend to the unrelated examples before it in the row.